In [1]:
import os
import time
from tqdm import tqdm

In [2]:
def search_ensembles(path, progress=True):
    file = open(path, "rb")
    file.seek(0, 2)
    file_size = file.tell()
    file.seek(0)
    ens_indexes = []
    current_offset = 0

    if progress:
        pbar = tqdm(total=file_size, unit='B', unit_scale=True, desc='Scanning file')
    else:
        pbar = None

    while True:
        file.seek(current_offset)
        header = file.read(4)

        if len(header) < 4:
            break

        if header[0] == 0x7f and header[1] == 0x7f:
            ens_size = header[2] + (header[3] << 8) + 2
            if 32 <= ens_size <= 4096:
                ens_indexes.append(current_offset)
                current_offset += ens_size
                if pbar:
                    pbar.update(ens_size)
                continue

        current_offset += 1
        if pbar:
            pbar.update(1)

    if pbar:
        pbar.close()

    return ens_indexes

In [4]:
path = "measurements.000"

start = time.time()
search_ensembles(path, progress=False)
no_progress_time = time.time() - start

start = time.time()
search_ensembles(path, progress=True)
with_progress_time = time.time() - start

print(f"\n⏱ Without progress bar: {no_progress_time:.2f} seconds")
print(f"⏱ With progress bar:    {with_progress_time:.2f} seconds")
print(f"📊 Overhead:            {with_progress_time - no_progress_time:.2f} seconds ({((with_progress_time / no_progress_time) - 1) * 100:.2f}%)")

Scanning file: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14.6M/14.6M [00:00<00:00, 752MB/s]


⏱ Without progress bar: 0.02 seconds
⏱ With progress bar:    0.02 seconds
📊 Overhead:            0.01 seconds (55.07%)
